# Ingest RAND Longitudinal SPSS Files to Databricks

**Source:** `D:\OneDrive\Documents\Databricks Training\HRS\RAND Longitudinal\randhrs1992_2022v1_SPSS`  

**Destination Volume:** `landing_catalog.hrs_vol`  
---

## Important Note
Databricks notebooks run on cloud compute and **cannot directly access your local PC files**. You must first upload files from your local directory to Databricks.


### UI Upload (Easiest and the method I used)

1. Download the randhrs1992_2020v1_spss file from the RAND HRS website
2. Open **Catalog Explorer** and create → `landing_catalog` → `external_data`
3. Click **Upload** button
4. Select `.sav` file(s) from your local directory
5. Wait for upload to complete
---


In [ ]:
%pip install pyreadstat
%restart_python

: 

In [0]:
# List source files to verify the upload
import os

volume_path = "/Volumes/landing_catalog/external_data/rand_hrs_raw_data/randhrs1992_2022v1.sav"

print("Files in volume:")
print("=" * 80)

files = dbutils.fs.ls(volume_path)
for file_info in files:
    size_mb = file_info.size / (1024 * 1024)
    print(f"{file_info.name:50s} {size_mb:>10.2f} MB")

print(f"\nTotal files: {len(files)}")

In [0]:
# ------------------------------------------------------------------------------
# HRS Bronze Ingestion
# Read SPSS (.sav) file in chunks and load to a Delta table.
#
# Purpose:
#   Performs a full refresh of the Bronze Delta table by dropping any existing
#   table and recreating it from the SPSS source file.
#
# Compatible with:
#   Databricks Free Edition
# ------------------------------------------------------------------------------

import time
import pyreadstat

# ------------------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------------------

sav_file = "/Volumes/landing_catalog/external_data/rand_hrs_raw_data/randhrs1992_2022v1.sav"
table_name = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"
chunk_size = 5000

# ------------------------------------------------------------------------------
# Start
# ------------------------------------------------------------------------------

start_time = time.time()

print("=" * 80)
print("HRS Bronze Data Load")
print("=" * 80)
print(f"Source File : {sav_file}")
print(f"Target Table: {table_name}")
print(f"Chunk Size  : {chunk_size:,}")
print()

# ------------------------------------------------------------------------------
# Remove existing table
# ------------------------------------------------------------------------------

print(f"Dropping table if it exists: {table_name}")
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# ------------------------------------------------------------------------------
# Create chunk reader
# ------------------------------------------------------------------------------

reader = pyreadstat.read_file_in_chunks(
    pyreadstat.read_sav,
    sav_file,
    chunksize=chunk_size
)

first_chunk = True
total_rows = 0

# ------------------------------------------------------------------------------
# Process chunks
# ------------------------------------------------------------------------------

try:

    for chunk_num, (pdf, meta) in enumerate(reader, start=1):

        rows = len(pdf)
        total_rows += rows

        print(f"Processing Chunk {chunk_num:,} ({rows:,} rows)")

        sdf = spark.createDataFrame(pdf)

        if first_chunk:

            sdf.write \
                .format("delta") \
                .saveAsTable(table_name)

            first_chunk = False

        else:

            sdf.write \
                .mode("append") \
                .format("delta") \
                .saveAsTable(table_name)

except Exception as ex:
    print()
    print("ERROR during data load.")
    print(ex)
    raise

# ------------------------------------------------------------------------------
# Validation
# ------------------------------------------------------------------------------

record_count = spark.table(table_name).count()

elapsed = time.time() - start_time

print()
print("=" * 80)
print("Load Complete")
print("=" * 80)
print(f"Chunks Processed : {chunk_num:,}")
print(f"Rows Read        : {total_rows:,}")
print(f"Rows Loaded      : {record_count:,}")
print(f"Elapsed Time     : {elapsed:.1f} seconds")
print("=" * 80)

In [0]:
%sql
-- Verify the table was created and preview data
SELECT * FROM dev_catalog.brz_raw_hrs.randhrs1992_2022v1 LIMIT 10